# KV cache: reuse past projections

Prerequisites: causal self-attention and autoregressive decoding. Estimated time: 30 minutes. Objectives: count recomputed versus cached K/V projections, quantify remaining history reads, and detect unsafe cache reuse.

A decoder computes keys and values for each prefix token. This notebook compares recomputing every prefix with appending each new token's K/V once. The toy counts projections; it does not model GPU latency or attention kernels.

## Assumptions and baseline

One sequence, one layer, one scalar K and V per token. A prompt has three tokens and decoding adds two. The baseline recomputes K/V for the entire visible prefix at every decode step. Both paths have the same three-token prompt prefill, which is excluded from this decode-only comparison.

In [ ]:
prefix = [2, 3, 5]
new_tokens = [7, 11]
baseline_projection_count = sum(len(prefix) + i for i in range(1, len(new_tokens) + 1))
assert baseline_projection_count == 9
print('baseline K/V projections:', baseline_projection_count)

## Cached decode

Prefill computes K/V for the prompt once. Each decode iteration computes K/V only for the newly generated token, then reuses the stored prefix. Queries and attention over stored history are still computed; caching does not make long-context attention constant-time.

In [ ]:
cache = [(token, token * 10) for token in prefix]
cached_new_projection_count = 0
history_lengths = []
for token in new_tokens:
    cache.append((token, token * 10))
    cached_new_projection_count += 1
    history_lengths.append(len(cache))
assert cache == [(t, t * 10) for t in prefix + new_tokens]
assert history_lengths == [4, 5]
assert cached_new_projection_count == 2
print('cached new-token projections:', cached_new_projection_count, 'history read lengths:', history_lengths)

## Comparison and metric

Across the two decode iterations after prompt prefill, caching reduces K/V projection work from nine token-projections to two (a 77.8% decode-only reduction). It does not claim a 77.8% end-to-end speedup: each new query still attends to a growing history, and real cost includes allocation and kernel behavior.

In [ ]:
reduction = 1 - cached_new_projection_count / baseline_projection_count
assert round(reduction, 3) == 0.778
assert sum(history_lengths) == 9  # cached attention still reads prior positions
print('projection reduction:', round(100 * reduction, 1), '%; cumulative history positions read:', sum(history_lengths))

## Edge and failure cases

An empty decode adds no entries. Reusing a cache for a different prefix is incorrect; cache identity must include the exact prefix and compatible model/configuration. Memory grows with live tokens even when projection work is saved.

In [ ]:
def append_token(cache, token, prefix_id, cache_prefix_id):
    if prefix_id != cache_prefix_id:
        raise ValueError('cache belongs to a different prefix')
    return cache + [(token, token * 10)]
assert append_token([], 13, 'same', 'same') == [(13, 130)]
try:
    append_token([], 13, 'new', 'old')
except ValueError:
    pass
else:
    raise AssertionError('must reject cross-prefix cache reuse')

## Exercises

1. Change the prompt to six tokens and generate four tokens. Compare recomputed K/V projections with cached new-token projections.
2. Extend the metric to two layers and explain which terms in KV memory scale with layers, heads, token count, and dtype.
3. Explain why per-token latency may still rise as the cache grows.

## Summary

KV caching stores prior per-layer keys and values so decode need not project the whole prefix again. New-token projections are incremental; attention still reads history. Validate cache identity, memory capacity, and cached-versus-full-prefix outputs in a real runtime. See the [concept note](../concepts/23-kv-cache.md) and the [practice assessment](../interview-prep/curriculum-practice.md#kv-cache).